# AD-vs-PD GWAS — what was run

AD (defined by neuropathology) vs PD across AMP-AD and AMP-PD, restricted to donors with WGS.
Five cohorts, four genotype callsets. Every command below was run on biowulf.

## Status

Per-callset steps — **all four callsets complete**:

| | wgs_harm | divco_hs | wb_dwgs | br_dsnwgs |
|---|---|---|---|---|
| 0 → pgen | done | done | shipped as pgen | done |
| 1 genotools | done | done | done | done |
| 2 normalize | done | done | done | done |

Cohort-level steps:

| step | state |
|---|---|
| 3 merge | done — 172,497,055 variants × 13,334 samples |
| 4 relatedness | done — 13,334/13,334 labelled, 11 ancestry strata |
| 5 excludelist | done — **12,495 retained**, 839 excluded |
| 6 ancestry QC + PCA (pass 1) | done, unfiltered — 6 of 11 strata have PCs |
| clinical §11–13 | done — `analysis_grain.csv` 12,495 rows, 17 of 44 contrasts viable |
| AF-concordance filter | running |
| 6 ancestry QC + PCA (pass 2) | next |
| 7 GWAS | not yet run on the four-callset cohort |

The AF filter is needed: unfiltered eta² is 0.984 on AJ's PC1 and 0.757 on EUR's PC2, and
removing br_dsnwgs changes neither — so it is real wgs_harm↔wb_dwgs structure, not the new
callset. (AAC 0.877 and AFR 0.708 are a different thing: one br_dsnwgs sample each, at 39.7σ
and 19.6σ. Neither stratum yields a viable contrast, so those PCs never reach the GWAS.)

## Setup

`config.sh` derives every path from its own location, so nothing here is machine-specific and
every command below stays short. Code is synced from the repo to the cluster; data never moves.

```bash
cd /data/CARDPB2/sysbio/wgs
source config.sh
```

## 0. VCF → pgen

Only br_dsnwgs needed this. Biallelic PASS SNPs, IDs set to `CHROM:POS:REF:ALT`, then
imported with the PAR split off chrX (a placeholder sex file only to permit the import —
real sex is applied in step 1).

```bash
bcftools view --apply-filters 'PASS,.' --min-alleles 2 --max-alleles 2 --type snps \
    $DIR_BR/joint_calls/AMPPD_postmortem_joint_gt_call_97donors.vcf.gz \
  | bcftools annotate --set-id '%CHROM:%POS:%REF:%ALT' \
  | bgzip -@ 8 > $DIR_BR/pgen/intermediate/br_dsnwgs_filtered.vcf.gz

plink2 --vcf $DIR_BR/pgen/intermediate/br_dsnwgs_filtered.vcf.gz \
    --chr 1-22,X,Y --split-par hg38 --update-sex placeholder_sex.txt \
    --make-pgen --out $RAW_BR
```

wgs_harm: 24 per-chromosome b37 VCFs → per-chrom filter → `--pmerge-list` → liftOver b37→hg38.
divco_hs: one pre-merged hg38 VCF, same two stages as above. wb_dwgs ships plink2 pfiles.

## Clinical §1–10

Reads the 11 per-cohort clinical files, derives `pheno` / `dx_detailed` per donor, resolves
genotype samples to donors, and writes the per-callset sex-update files step 1 needs.
§11–13 wait on cluster output and print SKIPPED on this pass.

```bash
python3 clinical_core.py
```

## 1. GenoTools — per-callset filter, ancestry, QC

Applies the sex file, filters to biallelic PASS SNPs, converts to bed, then projects onto the
GP2 reference panel for ancestry. Writes the predicted ancestry labels steps 4 and 6 read back.

```bash
./submit.sh scripts/01_genotools.sh --job-name=genotools_wgs_harm \
  --export=PGEN=$RAW_WGS,SEX_FILE=$(sex_file wgs_harm),OUT_DIR=$DIR_WGS/genotools,DATASET=wgs_harm

./submit.sh scripts/01_genotools.sh --job-name=genotools_divco_hs \
  --export=PGEN=$RAW_DC,SEX_FILE=$(sex_file divco_hs),OUT_DIR=$DIR_DC/genotools,DATASET=divco_hs

./submit.sh scripts/01_genotools.sh --job-name=genotools_wb_dwgs \
  --export=PGEN=$RAW_WB,SEX_FILE=$(sex_file wb_dwgs),OUT_DIR=$DIR_WB/genotools,DATASET=wb_dwgs

./submit.sh scripts/01_genotools.sh --job-name=genotools_br_dsnwgs \
  --export=PGEN=$RAW_BR,SEX_FILE=$(sex_file br_dsnwgs),OUT_DIR=$DIR_BR/genotools,DATASET=br_dsnwgs
```

## 2. Normalize

REF/ALT set from the GRCh38 reference and uniform `chr:pos:REF:ALT` IDs across all callsets,
so the merge in step 3 lines up. Output is bed. One job per callset; they are independent.

```bash
./submit.sh scripts/02_normalize.sh --job-name=norm_wgs --export=PFILE=$PF_WGS,OUT=$NORM_WGS,TAG=wgs
./submit.sh scripts/02_normalize.sh --job-name=norm_dc  --export=PFILE=$PF_DC,OUT=$NORM_DC,TAG=dc
./submit.sh scripts/02_normalize.sh --job-name=norm_wb  --export=PFILE=$PF_WB,OUT=$NORM_WB,TAG=wb
./submit.sh scripts/02_normalize.sh --job-name=norm_br  --export=PFILE=$PF_BR,OUT=$NORM_BR,TAG=br
```

The pass was validated on chr22 before being run genome-wide: the 3-way exact variant-ID overlap
went from ~83,905 to ~331,015, which is the allele-concordant ceiling.

## 3. Merge

plink1.9 union merge of the normalized filesets into one cohort (`cohort_merged`).

```bash
./submit.sh scripts/03_merge.sh
```

## 4. Relatedness

Cross-dataset KING on the common-variant set, run within each ancestry stratum using the
step 1 labels. Report-only — step 5 adjudicates.

```bash
./submit.sh scripts/04_relatedness.sh
```

## 5. Excludelist

Picks which of each duplicate/related pair to drop, joins the genotools QC failures, and writes
the retained-samples manifest. Text output only; no genotype file is modified.

```bash
./submit.sh scripts/05_excludelist.sh
```

## 6. Ancestry QC + PCA — **runs twice**

Per ancestry stratum: variant QC (`--geno 0.05 --maf 0.01 --hwe 1e-6`), LD-prune excluding
long-range-LD regions, then PCA. The pruned set is used for PCA only; the association set is
the full QC-passing variants.

Step 6 runs **twice**, with the AF-concordance filter built between the passes. Pass 1 must run
unfiltered, because that filter needs step 6's own output plus the grain — and step 6 cannot
depend on the grain without a circular ordering (grain ← §12 ← manifest ← step 6).

```bash
# pass 1 — unfiltered. Prints a NONE note for the missing exclusion list; expected.
./submit.sh scripts/06_ancestry_qc.sh

# is the filter needed? eta^2 = share of each PC's variance explained by source callset
python3 review/plot_pcs_by_callset.py

# only if it is. Then read the BY CALLSET PAIR table and the SENTINEL_LOCI tripwire in its log.
./submit.sh scripts/af_concordance_build.sh

# pass 2 — picks the exclusion list up automatically
./submit.sh scripts/06_ancestry_qc.sh
```

**What that filter removes, and why it is not simply "drop discordant variants."** It flags
variants whose allele frequency disagrees *between callsets*, compared only within an
(ancestry × diagnosis) cell so that disease is held constant. Without that constraint APOE would
be flagged for an entirely real reason — rs429358 genuinely differs between the AD-source and
PD-source callsets — and the study's strongest locus would be deleted. On the 3-callset cohort,
147 variants flagged in EUR at |dAF| > 0.10 took AJ's callset eta^2 from 0.748 to 0.055, where
147 *random* variants left it at 0.738. The list applies to the association set, not just the
PCA input: a variant mismapped badly enough to bend PC1 also produces spurious associations that
no PC covariate can reach.

## Clinical §11–13

Second pass, after step 6. §11 labels the QC outcomes by reason; §12 reconciles ancestry and PCs
into `analysis_grain.csv` — one row per genome, and the input to the GWAS. Because §12 carries
the PCs, the grain must be rebuilt whenever step 6 re-runs.

```bash
python3 clinical_core.py
```

## 7. GWAS

Per ancestry × per contrast, `plink2 --glm` with Firth fallback, covariates sex + PC1–PC10.
A differential-missingness filter runs per contrast before the association. Contrasts include
PD-vs-AD, each vs other diagnoses and controls, the within-cohort pairs, and a control-vs-control
scan for batch artifacts.

```bash
./submit.sh scripts/07_gwas.sh
```

## 8. Plots

QQ and Manhattan per contrast; the mask applies the control-vs-control hits to the disease
sumstats.

```bash
python3 review/plot_gwas.py
python3 review/mask_cohort_artifacts.py
```